In [ ]:
# =============================================================================
#  Publication statistics: BOB vs EPN vs PEAP  (per-subject, paired)
#  Per metric:
#    (1) Friedman omnibus            + Kendall's W (omnibus effect size)
#    (2) Pairwise difference test    Wilcoxon signed-rank, Holm-corrected
#                                    + 95% CI (two-sided) + rank-biserial r
#                                    + Cohen's dz with 95% bootstrap CI
#    (3) Pairwise equivalence (TOST) @ alpha, pre-registered SESOI, 90% CI
#  Emits a tidy results DataFrame (-> CSV / LaTeX).
#  Single Colab cell. Reads JSON from Google Drive.
# =============================================================================
import json, os, itertools
import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------- CONFIG ----------------------------------------
# Files at  MyDrive/result_alpha/{MODEL}_lemon/{model}_alpha_results.json
DRIVE_DIR = "/content/drive/MyDrive/RAIN_IRC/results"
FILE_TAG  = {"BOB": "bob", "EPN": "epn", "PEAP": "peap"}   # filename stem per model
FILES = {m: os.path.join(DRIVE_DIR, f"{m}_onset", f"{tag}_theta_results.json")
         for m, tag in FILE_TAG.items()}
METRIC_BLOCK = {"BOB": "opt_metrics_pooled",               # BOB: optimized condition
                "EPN": "metrics_pooled", "PEAP": "metrics_pooled"}

PRIMARY   = "mace"
SECONDARY = ["acc", "csd"]
ALPHA     = 0.05
N_BOOT    = 10000
SEED      = 0

# --- SESOI: pre-registered smallest effect of interest, from a 1 ms stimulation-
#     timing tolerance at the band-centre frequency (NOT from observed gaps).
#     Phase advances 0.36*f deg/ms, so the margin is band-specific.
#     THIS SCRIPT = evoked theta (see DRIVE_DIR / *_theta_results). Mu run: F_CENTRE_HZ = 11.0.
TIMING_TOL_MS = 1.0
F_CENTRE_HZ   = 6.0                                          # theta (4-8 Hz) centre; mu = 11.0
SESOI_MACE    = TIMING_TOL_MS * 1e-3 * F_CENTRE_HZ * 360.0   # theta: 2.16 deg | mu: 3.96 deg
# ACC = 100 - MACE/1.8 exactly => ACC margin = MACE margin / 1.8, and the ACC
# equivalence test is mathematically identical to MACE's (report MACE as primary).
SESOI = {"mace": SESOI_MACE, "acc": SESOI_MACE / 1.8, "csd": 4.0}
UNITS = {"mace": "deg", "csd": "deg", "acc": "pp", "plv": "", "bias": "deg"}
OUT_DIR = "."                       # where CSVs are written
# -----------------------------------------------------------------------------

# ---- mount Drive & load ------------------------------------------------------
def _ensure_files():
    if not os.path.isdir(DRIVE_DIR):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:
            raise RuntimeError(f"Could not mount Drive: {e}")
    missing = [p for p in FILES.values() if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError("Not found (check DRIVE_DIR / FILE_TAG):\n  " +
                                "\n  ".join(missing))

_ensure_files()
raw = {m: json.load(open(p)) for m, p in FILES.items()}
maps = {m: ({s["name"]: s for s in raw[m]["subjects"]}, METRIC_BLOCK[m])
        for m in FILES}
names = sorted(set.intersection(*[set(maps[m][0]) for m in FILES]))
assert names, "No common subjects."
N = len(names)
print(f"N = {N} paired subjects\n")

def vec(model, metric):
    sm, blk = maps[model]
    return np.array([sm[n][blk][metric] for n in names], float)

# ---- statistics --------------------------------------------------------------
def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, idx in enumerate(order):
        run = max(run, (m - rank) * p[idx]); adj[idx] = min(run, 1.0)
    return adj

def kendalls_w(arrs):
    """Kendall's W from Friedman: W = chi2 / (N (k-1)). Range [0,1]."""
    chi2, p = stats.friedmanchisquare(*arrs)
    k = len(arrs); n = len(arrs[0])
    W = chi2 / (n * (k - 1))
    return chi2, k - 1, p, W

def cohen_dz(d):
    return d.mean() / d.std(ddof=1)

def dz_boot_ci(d, conf=0.95):
    rng = np.random.default_rng(SEED)
    n = len(d)
    idx = rng.integers(0, n, size=(N_BOOT, n))
    samp = d[idx]
    dz = samp.mean(1) / samp.std(1, ddof=1)
    lo, hi = np.percentile(dz, [100*(1-conf)/2, 100*(1+conf)/2])
    return lo, hi

def rank_biserial(d):
    """Matched-pairs rank-biserial r: (W+ - W-)/(W+ + W-)."""
    d = d[d != 0.0]
    if d.size == 0:
        return 0.0
    ranks = stats.rankdata(np.abs(d))
    wp, wm = ranks[d > 0].sum(), ranks[d < 0].sum()
    return (wp - wm) / (wp + wm)

def diff_ci(d, conf=0.95):
    """Two-sided descriptive CI on paired-difference mean (pairs w/ Wilcoxon)."""
    n = len(d); md, se = d.mean(), d.std(ddof=1)/np.sqrt(n)
    t = stats.t.ppf(1 - (1-conf)/2, n-1)
    return md - t*se, md + t*se

def paired_tost(d, sesoi):
    """TOST @ ALPHA. Decision interval = 90% CI (= 100(1-2a)%). Returns
       (mean, p_tost, ci90_lo, ci90_hi, equivalent)."""
    n = len(d); md, se = d.mean(), d.std(ddof=1)/np.sqrt(n); df = n-1
    p = max(stats.t.sf((md+sesoi)/se, df), stats.t.cdf((md-sesoi)/se, df))
    tc = stats.t.ppf(1-ALPHA, df)
    lo, hi = md - tc*se, md + tc*se
    return md, p, lo, hi, (p < ALPHA and lo > -sesoi and hi < sesoi)

# ---- run per metric ----------------------------------------------------------
pairs = list(itertools.combinations(FILES.keys(), 2))
omni_rows, pair_rows = [], []

def run_metric(metric):
    u = UNITS.get(metric, ""); s = SESOI.get(metric)
    data = {m: vec(m, metric) for m in FILES}
    print("=" * 80)
    print(f"METRIC: {metric.upper()} ({u})   SESOI (pre-registered) = +/-{s} {u}")
    print("=" * 80)
    for m in FILES:
        v = data[m]
        print(f"  {m:5s}  mean {v.mean():7.3f}  sd {v.std(ddof=1):6.3f}  "
              f"median {np.median(v):7.3f}")

    # (1) omnibus
    chi2, df, p_om, W = kendalls_w([data[m] for m in FILES])
    print(f"\n  (1) Friedman:  chi2({df}) = {chi2:.3f}, p = {p_om:.3g}   "
          f"Kendall's W = {W:.3f}")
    omni_rows.append(dict(metric=metric, chi2=chi2, df=df, p=p_om, kendall_w=W))

    # (2) difference + (3) equivalence
    wil_p, tmp = [], []
    for a, b in pairs:
        d = data[a] - data[b]
        try:    Wstat, pw = stats.wilcoxon(data[a], data[b])
        except ValueError: Wstat, pw = np.nan, 1.0
        wil_p.append(pw)
        tmp.append((a, b, d, Wstat, pw))
    wil_adj = holm(wil_p)

    print(f"\n  (2) Difference (Wilcoxon, Holm; 95% CI two-sided; "
          f"dz 95% bootstrap CI; r_rb = rank-biserial):")
    print(f"  {'pair':10s}{'diff':>8s}{'  95% CI':>18s}  {'dz':>6s}"
          f"{'  dz 95% CI':>16s}{'r_rb':>7s}{'p_holm':>9s}")
    for (a, b, d, Wstat, pw), padj in zip(tmp, wil_adj):
        md = d.mean(); c95 = diff_ci(d); dz = cohen_dz(d); dzc = dz_boot_ci(d)
        rrb = rank_biserial(d); excl = "!" if not (c95[0] <= 0 <= c95[1]) else " "
        flag = "*" if padj < ALPHA else " "
        print(f"  {a+'-'+b:10s}{md:8.3f} [{c95[0]:7.3f},{c95[1]:7.3f}]{excl}"
              f"{dz:6.2f} [{dzc[0]:6.2f},{dzc[1]:6.2f}]{rrb:7.2f}{padj:9.3g}{flag}")

    print(f"\n  (3) Equivalence (paired TOST @ alpha={ALPHA}; "
          f"90% CI = decision interval):")
    print(f"  {'pair':10s}{'diff':>8s}{'      90% CI':>20s}{'p_TOST':>10s}  verdict")
    for (a, b, d, _, _) in tmp:
        md, pt, lo, hi, eq = paired_tost(d, s)
        verdict = "EQUIVALENT" if eq else "inconclusive"
        print(f"  {a+'-'+b:10s}{md:8.3f}  [{lo:7.3f},{hi:7.3f}]{pt:10.3g}  {verdict}")

    # tidy rows
    for (a, b, d, Wstat, pw), padj in zip(tmp, wil_adj):
        md = d.mean(); c95 = diff_ci(d); dz = cohen_dz(d); dzc = dz_boot_ci(d)
        rrb = rank_biserial(d); _, pt, lo, hi, eq = paired_tost(d, s)
        pair_rows.append(dict(
            metric=metric, pair=f"{a}-{b}", mean_diff=md,
            ci95_lo=c95[0], ci95_hi=c95[1], wilcoxon_W=Wstat,
            p_raw=pw, p_holm=padj, sig=bool(padj < ALPHA),
            rank_biserial=rrb, cohen_dz=dz, dz_ci_lo=dzc[0], dz_ci_hi=dzc[1],
            sesoi=s, tost_ci90_lo=lo, tost_ci90_hi=hi, p_tost=pt,
            equivalent=bool(eq)))
    print()

print("PRIMARY ENDPOINT")
run_metric(PRIMARY)
print("SECONDARY ENDPOINTS")
for met in SECONDARY:
    run_metric(met)

# ---- tidy outputs ------------------------------------------------------------
omni_df = pd.DataFrame(omni_rows)
pair_df = pd.DataFrame(pair_rows)
omni_path = os.path.join(OUT_DIR, "phase_stats_omnibus.csv")
pair_path = os.path.join(OUT_DIR, "phase_stats_pairwise.csv")
omni_df.to_csv(omni_path, index=False)
pair_df.to_csv(pair_path, index=False)
print("Saved:", omni_path, "and", pair_path)

# LaTeX (primary metric pairwise) -- paste-ready skeleton
prim_tbl = pair_df[pair_df.metric == PRIMARY][
    ["pair", "mean_diff", "ci95_lo", "ci95_hi",
     "cohen_dz", "p_holm", "tost_ci90_lo", "tost_ci90_hi", "equivalent"]]
print("\n--- LaTeX (primary =", PRIMARY, ") ---")
print(prim_tbl.to_latex(index=False, float_format="%.3f"))

N = 20 paired subjects

PRIMARY ENDPOINT
METRIC: MACE (deg)   SESOI (pre-registered) = +/-2.16 deg
  BOB    mean  48.396  sd  3.491  median  47.566
  EPN    mean  48.220  sd  3.873  median  48.701
  PEAP   mean  51.458  sd  4.182  median  52.286

  (1) Friedman:  chi2(2) = 19.600, p = 5.55e-05   Kendall's W = 0.490

  (2) Difference (Wilcoxon, Holm; 95% CI two-sided; dz 95% bootstrap CI; r_rb = rank-biserial):
  pair          diff            95% CI      dz       dz 95% CI   r_rb   p_holm
  BOB-EPN      0.176 [ -0.509,  0.862]   0.12 [ -0.28,  0.91]   0.38    0.143 
  BOB-PEAP    -3.062 [ -4.000, -2.123]! -1.53 [ -2.32, -1.08]  -0.98 1.72e-05*
  EPN-PEAP    -3.238 [ -4.499, -1.977]! -1.20 [ -2.27, -0.69]  -0.90 0.000164*

  (3) Equivalence (paired TOST @ alpha=0.05; 90% CI = decision interval):
  pair          diff              90% CI    p_TOST  verdict
  BOB-EPN      0.176  [ -0.390,  0.742]  3.97e-06  EQUIVALENT
  BOB-PEAP    -3.062  [ -3.837, -2.286]     0.971  inconclusive
  EPN-PEA

In [ ]:
# =============================================================================
#  Publication statistics: BOB vs EPN vs PEAP  (per-subject, paired)
#  Per metric:
#    (1) Friedman omnibus            + Kendall's W (omnibus effect size)
#    (2) Pairwise difference test    Wilcoxon signed-rank, Holm-corrected
#                                    + 95% CI (two-sided) + rank-biserial r
#                                    + Cohen's dz with 95% bootstrap CI
#    (3) Pairwise equivalence (TOST) @ alpha, pre-registered SESOI, 90% CI
#  Emits a tidy results DataFrame (-> CSV / LaTeX).
#  Single Colab cell. Reads JSON from Google Drive.
# =============================================================================
import json, os, itertools
import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------- CONFIG ----------------------------------------
# Files at  MyDrive/result_alpha/{MODEL}_lemon/{model}_alpha_results.json
DRIVE_DIR = "/content/drive/MyDrive/RAIN_IRC/result_alpha"
FILE_TAG  = {"BOB": "bob", "EPN": "epn", "PEAP": "peap"}   # filename stem per model
FILES = {m: os.path.join(DRIVE_DIR, f"{m}_lemon", f"{tag}_alpha_results.json")
         for m, tag in FILE_TAG.items()}
METRIC_BLOCK = {"BOB": "opt_metrics_pooled",               # BOB: optimized condition
                "EPN": "metrics_pooled", "PEAP": "metrics_pooled"}

PRIMARY   = "mace"
SECONDARY = ["acc", "csd"]
ALPHA     = 0.05
N_BOOT    = 10000
SEED      = 0
# --- SESOI: pre-registered smallest effect of interest, from a 1 ms stimulation-
#     timing tolerance at the band-centre frequency (NOT from observed gaps).
#     Phase advances 0.36*f deg/ms, so the margin is band-specific.
#     THIS SCRIPT = evoked theta (see DRIVE_DIR / *_theta_results). Mu run: F_CENTRE_HZ = 11.0.
TIMING_TOL_MS = 1.0
F_CENTRE_HZ   = 11.0                                          # theta (4-8 Hz) centre; mu = 11.0
SESOI_MACE    = TIMING_TOL_MS * 1e-3 * F_CENTRE_HZ * 360.0   # theta: 2.16 deg | mu: 3.96 deg
# ACC = 100 - MACE/1.8 exactly => ACC margin = MACE margin / 1.8, and the ACC
# equivalence test is mathematically identical to MACE's (report MACE as primary).
SESOI = {"mace": SESOI_MACE, "acc": SESOI_MACE / 1.8, "csd": 4.0}
UNITS = {"mace": "deg", "csd": "deg", "acc": "pp", "plv": "", "bias": "deg"}
OUT_DIR = "."                       # where CSVs are written
# -----------------------------------------------------------------------------

# ---- mount Drive & load ------------------------------------------------------
def _ensure_files():
    if not os.path.isdir(DRIVE_DIR):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:
            raise RuntimeError(f"Could not mount Drive: {e}")
    missing = [p for p in FILES.values() if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError("Not found (check DRIVE_DIR / FILE_TAG):\n  " +
                                "\n  ".join(missing))

_ensure_files()
raw = {m: json.load(open(p)) for m, p in FILES.items()}
maps = {m: ({s["name"]: s for s in raw[m]["subjects"]}, METRIC_BLOCK[m])
        for m in FILES}
names = sorted(set.intersection(*[set(maps[m][0]) for m in FILES]))
assert names, "No common subjects."
N = len(names)
print(f"N = {N} paired subjects\n")

def vec(model, metric):
    sm, blk = maps[model]
    return np.array([sm[n][blk][metric] for n in names], float)

# ---- statistics --------------------------------------------------------------
def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, idx in enumerate(order):
        run = max(run, (m - rank) * p[idx]); adj[idx] = min(run, 1.0)
    return adj

def kendalls_w(arrs):
    """Kendall's W from Friedman: W = chi2 / (N (k-1)). Range [0,1]."""
    chi2, p = stats.friedmanchisquare(*arrs)
    k = len(arrs); n = len(arrs[0])
    W = chi2 / (n * (k - 1))
    return chi2, k - 1, p, W

def cohen_dz(d):
    return d.mean() / d.std(ddof=1)

def dz_boot_ci(d, conf=0.95):
    rng = np.random.default_rng(SEED)
    n = len(d)
    idx = rng.integers(0, n, size=(N_BOOT, n))
    samp = d[idx]
    dz = samp.mean(1) / samp.std(1, ddof=1)
    lo, hi = np.percentile(dz, [100*(1-conf)/2, 100*(1+conf)/2])
    return lo, hi

def rank_biserial(d):
    """Matched-pairs rank-biserial r: (W+ - W-)/(W+ + W-)."""
    d = d[d != 0.0]
    if d.size == 0:
        return 0.0
    ranks = stats.rankdata(np.abs(d))
    wp, wm = ranks[d > 0].sum(), ranks[d < 0].sum()
    return (wp - wm) / (wp + wm)

def diff_ci(d, conf=0.95):
    """Two-sided descriptive CI on paired-difference mean (pairs w/ Wilcoxon)."""
    n = len(d); md, se = d.mean(), d.std(ddof=1)/np.sqrt(n)
    t = stats.t.ppf(1 - (1-conf)/2, n-1)
    return md - t*se, md + t*se

def paired_tost(d, sesoi):
    """TOST @ ALPHA. Decision interval = 90% CI (= 100(1-2a)%). Returns
       (mean, p_tost, ci90_lo, ci90_hi, equivalent)."""
    n = len(d); md, se = d.mean(), d.std(ddof=1)/np.sqrt(n); df = n-1
    p = max(stats.t.sf((md+sesoi)/se, df), stats.t.cdf((md-sesoi)/se, df))
    tc = stats.t.ppf(1-ALPHA, df)
    lo, hi = md - tc*se, md + tc*se
    return md, p, lo, hi, (p < ALPHA and lo > -sesoi and hi < sesoi)

# ---- run per metric ----------------------------------------------------------
pairs = list(itertools.combinations(FILES.keys(), 2))
omni_rows, pair_rows = [], []

def run_metric(metric):
    u = UNITS.get(metric, ""); s = SESOI.get(metric)
    data = {m: vec(m, metric) for m in FILES}
    print("=" * 80)
    print(f"METRIC: {metric.upper()} ({u})   SESOI (pre-registered) = +/-{s} {u}")
    print("=" * 80)
    for m in FILES:
        v = data[m]
        print(f"  {m:5s}  mean {v.mean():7.3f}  sd {v.std(ddof=1):6.3f}  "
              f"median {np.median(v):7.3f}")

    # (1) omnibus
    chi2, df, p_om, W = kendalls_w([data[m] for m in FILES])
    print(f"\n  (1) Friedman:  chi2({df}) = {chi2:.3f}, p = {p_om:.3g}   "
          f"Kendall's W = {W:.3f}")
    omni_rows.append(dict(metric=metric, chi2=chi2, df=df, p=p_om, kendall_w=W))

    # (2) difference + (3) equivalence
    wil_p, tmp = [], []
    for a, b in pairs:
        d = data[a] - data[b]
        try:    Wstat, pw = stats.wilcoxon(data[a], data[b])
        except ValueError: Wstat, pw = np.nan, 1.0
        wil_p.append(pw)
        tmp.append((a, b, d, Wstat, pw))
    wil_adj = holm(wil_p)

    print(f"\n  (2) Difference (Wilcoxon, Holm; 95% CI two-sided; "
          f"dz 95% bootstrap CI; r_rb = rank-biserial):")
    print(f"  {'pair':10s}{'diff':>8s}{'  95% CI':>18s}  {'dz':>6s}"
          f"{'  dz 95% CI':>16s}{'r_rb':>7s}{'p_holm':>9s}")
    for (a, b, d, Wstat, pw), padj in zip(tmp, wil_adj):
        md = d.mean(); c95 = diff_ci(d); dz = cohen_dz(d); dzc = dz_boot_ci(d)
        rrb = rank_biserial(d); excl = "!" if not (c95[0] <= 0 <= c95[1]) else " "
        flag = "*" if padj < ALPHA else " "
        print(f"  {a+'-'+b:10s}{md:8.3f} [{c95[0]:7.3f},{c95[1]:7.3f}]{excl}"
              f"{dz:6.2f} [{dzc[0]:6.2f},{dzc[1]:6.2f}]{rrb:7.2f}{padj:9.3g}{flag}")

    print(f"\n  (3) Equivalence (paired TOST @ alpha={ALPHA}; "
          f"90% CI = decision interval):")
    print(f"  {'pair':10s}{'diff':>8s}{'      90% CI':>20s}{'p_TOST':>10s}  verdict")
    for (a, b, d, _, _) in tmp:
        md, pt, lo, hi, eq = paired_tost(d, s)
        verdict = "EQUIVALENT" if eq else "inconclusive"
        print(f"  {a+'-'+b:10s}{md:8.3f}  [{lo:7.3f},{hi:7.3f}]{pt:10.3g}  {verdict}")

    # tidy rows
    for (a, b, d, Wstat, pw), padj in zip(tmp, wil_adj):
        md = d.mean(); c95 = diff_ci(d); dz = cohen_dz(d); dzc = dz_boot_ci(d)
        rrb = rank_biserial(d); _, pt, lo, hi, eq = paired_tost(d, s)
        pair_rows.append(dict(
            metric=metric, pair=f"{a}-{b}", mean_diff=md,
            ci95_lo=c95[0], ci95_hi=c95[1], wilcoxon_W=Wstat,
            p_raw=pw, p_holm=padj, sig=bool(padj < ALPHA),
            rank_biserial=rrb, cohen_dz=dz, dz_ci_lo=dzc[0], dz_ci_hi=dzc[1],
            sesoi=s, tost_ci90_lo=lo, tost_ci90_hi=hi, p_tost=pt,
            equivalent=bool(eq)))
    print()

print("PRIMARY ENDPOINT")
run_metric(PRIMARY)
print("SECONDARY ENDPOINTS")
for met in SECONDARY:
    run_metric(met)

# ---- tidy outputs ------------------------------------------------------------
omni_df = pd.DataFrame(omni_rows)
pair_df = pd.DataFrame(pair_rows)
omni_path = os.path.join(OUT_DIR, "phase_stats_omnibus.csv")
pair_path = os.path.join(OUT_DIR, "phase_stats_pairwise.csv")
omni_df.to_csv(omni_path, index=False)
pair_df.to_csv(pair_path, index=False)
print("Saved:", omni_path, "and", pair_path)

# LaTeX (primary metric pairwise) -- paste-ready skeleton
prim_tbl = pair_df[pair_df.metric == PRIMARY][
    ["pair", "mean_diff", "ci95_lo", "ci95_hi",
     "cohen_dz", "p_holm", "tost_ci90_lo", "tost_ci90_hi", "equivalent"]]
print("\n--- LaTeX (primary =", PRIMARY, ") ---")
print(prim_tbl.to_latex(index=False, float_format="%.3f"))

N = 20 paired subjects

PRIMARY ENDPOINT
METRIC: MACE (deg)   SESOI (pre-registered) = +/-3.96 deg
  BOB    mean  28.276  sd  5.201  median  27.945
  EPN    mean  27.468  sd  5.401  median  26.495
  PEAP   mean  28.246  sd  4.950  median  27.345

  (1) Friedman:  chi2(2) = 8.400, p = 0.015   Kendall's W = 0.210

  (2) Difference (Wilcoxon, Holm; 95% CI two-sided; dz 95% bootstrap CI; r_rb = rank-biserial):
  pair          diff            95% CI      dz       dz 95% CI   r_rb   p_holm
  BOB-EPN      0.807 [  0.351,  1.263]!  0.83 [  0.47,  1.35]   0.77   0.0043*
  BOB-PEAP     0.029 [ -0.491,  0.550]   0.03 [ -0.43,  0.52]   0.07    0.812 
  EPN-PEAP    -0.778 [ -1.353, -0.202]! -0.63 [ -1.19, -0.23]  -0.61   0.0306*

  (3) Equivalence (paired TOST @ alpha=0.05; 90% CI = decision interval):
  pair          diff              90% CI    p_TOST  verdict
  BOB-EPN      0.807  [  0.430,  1.184]   5.2e-12  EQUIVALENT
  BOB-PEAP     0.029  [ -0.400,  0.459]  1.09e-12  EQUIVALENT
  EPN-PEAP    -